In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)


In [2]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi")


AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '5e140f73-7fbe-4dba-b557-617c31cf9b7d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 02 May 2026 17:07:50 GMT', 'content-type': 'application/json', 'content-length': '234', 'connection': 'keep-alive', 'x-amzn-requestid': '5e140f73-7fbe-4dba-b557-617c31cf9b7d'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [397]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'}, id='lc_run--019de9a9-265d-71a0-a9f4-c56e216173f4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1, 'output_tokens': 9, 'total_tokens': 10, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [7]:
# External store for session memory
store = {}
# Define a buffer window message handler using trim_messages
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # Initialize empty session memory if needed
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    # Load current messages
    messages = store[session_id].messages
    # Trim messages manually like a buffer window
    trimmed = trim_messages(
        messages=messages,
        token_counter=llm,        
        max_tokens=30,            
        strategy="last",          # Keep most recent messages
        start_on="human",         # Ensure chat starts properly
        include_system=True,      # Keep SystemMessage if present
        allow_partial=False       # Don't allow modway trimming of a message 
    )
    # Re-store trimmed messages
    store[session_id] = InMemoryChatMessageHistory(messages=trimmed)
    return store[session_id]


In [6]:
chain = RunnableWithMessageHistory(llm, get_session_history)
# Define session ID
session_id = "window_memory_session"

In [8]:
response1 = chain.invoke("Hi there!", config={"configurable": {"session_id": session_id}})
print("Response 1:", response1)
print("_____" * 20)
response2 = chain.invoke("Tell me a programming joke.", config={"configurable": {"session_id": session_id}})
print("Response 2:", response2)
print("_____" * 20)
response3 = chain.invoke("What is the capital of Japan?", config={"configurable": {"session_id": session_id}})
print("Response 3:", response3)
print("_____" * 20)
# Print the trimmed message history (i.e., buffer window)
print("ðŸ“œ Final Trimmed Buffer Window:")
for msg in store[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")


Response 1: content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': 'fa20fde8-edc7-48d1-8895-0cfad8684737', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 02 May 2026 17:10:24 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': 'fa20fde8-edc7-48d1-8895-0cfad8684737'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [383]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019de9ab-808f-7321-b1b2-8ab03fe9b654-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 9, 'total_tokens': 12, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
____________________________________________________________________________________________________


/home/labuser/.local/lib/python3.10/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Response 2: content="Why did the programmer quit their job?\n\nBecause they couldn't *code*.\n\n(Get it? They couldn't bear to *co-exist* with bad code!)" additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '877c9e8d-3be5-4117-909c-c3044f274f17', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 02 May 2026 17:10:34 GMT', 'content-type': 'application/json', 'content-length': '335', 'connection': 'keep-alive', 'x-amzn-requestid': '877c9e8d-3be5-4117-909c-c3044f274f17'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [1112]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019de9ab-a5fe-7050-9e09-9baa318ead38-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 37, 'total_tokens': 55, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
____________________________________________________________________________________________________
Response 